# Wang 2024, Case 1: 7-activity house build

Reproduces the network from Case 1 of [Wang, Wang & Jin (2024)](https://doi.org/10.1016/j.cie.2024.110603), originally drawn from *Home Building Answers* (2008). The project consists of 7 home-construction activities arranged as a chain with one parallel branch. During execution, activities A2, A3, A5 and A6 slipped, producing a 5-day overall project delay.

This notebook shows how `shapcpm` attributes the **total project makespan** across activities for both the planned and actual schedules. The closing section discusses the relationship to the *delay* allocation reported in Wang 2024 — they are Shapley values of different value functions and answer different questions.

In [1]:
from shapcpm import CPMNetworkBuilder

## Network description

| Node | Activity                | Planned (days) | Actual (days) |
|------|-------------------------|---------------:|--------------:|
| A1   | Clean lot               | 1              | 1             |
| A2   | Pour footings           | 1              | 2             |
| A3   | Frame house             | 5              | 7             |
| A4   | Insulate                | 1              | 1             |
| A5   | Drywall                 | 3              | 5             |
| A6   | Exterior siding/roofing | 2              | 3             |
| A7   | Paint                   | 1              | 1             |

Topology (Wang 2024, Fig. 5):

```
A1 → A2 → A3 → A4 ─┬→ A5 ─┬→ A7
                  └→ A6 ─┘
```

A4 fans out into two parallel branches (`A5` and `A6`) which both feed into the final paint activity `A7`.

## Planned schedule

Build the network with the planned (a-column) durations and inspect the critical path.

In [2]:
planned = (
    CPMNetworkBuilder()
    .add_task("A1", 1)
    .add_task("A2", 1)
    .add_task("A3", 5)
    .add_task("A4", 1)
    .add_task("A5", 3)
    .add_task("A6", 2)
    .add_task("A7", 1)
    .add_dependency("A2", "A1")
    .add_dependency("A3", "A2")
    .add_dependency("A4", "A3")
    .add_dependency("A5", "A4")
    .add_dependency("A6", "A4")
    .add_dependency("A7", "A5")
    .add_dependency("A7", "A6")
    .build()
)

print("Critical path:", planned.get_critical_path())
print("Planned makespan:", planned.get_last_end_time(), "days")

Critical path: ['A1', 'A2', 'A3', 'A4', 'A5', 'A7']
Planned makespan: 12 days


Expected: critical path `A1 → A2 → A3 → A4 → A5 → A7`, makespan `12` days. The parallel `A6` branch totals 11 days and carries 1 day of slack.

## Actual schedule

Now build the network with the actual (b-column) durations. The critical path stays on the same branch but the makespan grows by 5 days.

In [3]:
actual = (
    CPMNetworkBuilder()
    .add_task("A1", 1)
    .add_task("A2", 2)
    .add_task("A3", 7)
    .add_task("A4", 1)
    .add_task("A5", 5)
    .add_task("A6", 3)
    .add_task("A7", 1)
    .add_dependency("A2", "A1")
    .add_dependency("A3", "A2")
    .add_dependency("A4", "A3")
    .add_dependency("A5", "A4")
    .add_dependency("A6", "A4")
    .add_dependency("A7", "A5")
    .add_dependency("A7", "A6")
    .build()
)

print("Critical path:", actual.get_critical_path())
print("Actual makespan:", actual.get_last_end_time(), "days")
print(
    "Project delay:",
    actual.get_last_end_time() - planned.get_last_end_time(),
    "days"
)

Critical path: ['A1', 'A2', 'A3', 'A4', 'A5', 'A7']
Actual makespan: 17 days
Project delay: 5 days


## Exact Shapley attribution of the actual makespan

`get_shapley_values_exact()` allocates the full 17-day makespan across the seven activities. Tasks with longer durations and tasks that lie on the critical path more often carry larger shares; the values sum exactly to the makespan.

In [4]:
exact = actual.get_shapley_values_exact()
for task, value in exact.items():
    print(f"{task}: {value:.3f}")
print(f"Sum: {sum(exact.values()):.3f} (= makespan)")

A1: 1.000
A2: 2.000
A3: 7.000
A4: 1.000
A5: 3.500
A6: 1.500
A7: 1.000
Sum: 17.000 (= makespan)


## Approximate Shapley attribution

For larger networks the Monte Carlo solver is more practical. With 2000 samples on a 7-activity network the values converge close to the exact answer.

In [5]:
approx = actual.get_shapley_values_approx(num_samples=2000)
for task, value in approx.items():
    print(f"{task}: {value:.3f}")
print(f"Sum: {sum(approx.values()):.3f}")

A1: 1.000
A2: 2.000
A3: 7.000
A4: 1.000
A5: 3.512
A6: 1.488
A7: 1.000
Sum: 17.000


## Relationship to Wang 2024's delay allocation

Wang, Wang & Jin (2024) attribute the *project delay* `T̂ − T*` (5 days), not the total makespan. For Case 1 they report:

| Task | Wang allocation (days) |
|------|-----------------------:|
| A2   | 1                      |
| A3   | 2                      |
| A5   | 2                      |
| A6   | 0 (within slack)       |
| others | 0                    |

Wang's allocation comes from a Shapley game whose value function is `T'(S) = makespan(S delayed, rest planned) − T*` — i.e. coalitions toggle each activity between its planned and actual duration. `shapcpm`'s value function is different: it toggles each activity between its current duration and zero, so it answers "how much of the total makespan does each activity contribute?" rather than "how much of the slip does each delayed activity contribute?".

The two questions share the same Shapley axioms but produce different per-activity numbers. Subtracting `Shapley(actual) − Shapley(planned)` gives a vector that sums to the project delay (5 days) but does not in general match Wang's allocation because the underlying coalitional games differ.

If you need exact Wang-style delay attribution, the recommended path today is to build a network whose makespan equals the project delay (e.g. by encoding each activity's slip as a separate task in series with a fixed planned scaffold) and run `shapcpm` on it.